# Split Purchase Detection Notebook

This notebook identifies purchase orders that appear intentionally split to avoid SAP MM release strategy approval thresholds.

## 1. Load PO data

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", palette="Set2")
pd.options.display.float_format = "{:,.0f}".format

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "03_Data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "03_Data"
OUTPUT_DIR = PROJECT_ROOT / "05_Python_Analysis" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def yen(value):
    return f"¥{value:,.0f}"

purchase_orders = pd.read_csv(DATA_DIR / "purchase_orders.csv", parse_dates=["po_date", "approval_date"])
vendors = pd.read_csv(DATA_DIR / "vendors.csv", parse_dates=["registration_date"])
po_data = purchase_orders.merge(vendors[["vendor_id", "vendor_name", "vendor_category", "is_ghost_vendor"]], on="vendor_id", how="left")
po_data.head()

## 2. Define split purchase logic

In [ ]:
APPROVAL_THRESHOLD_JPY = 500_000
MIN_CLUSTER_COUNT = 2

# SAP business logic: separate POs to the same vendor on the same day can circumvent a release threshold.
below_threshold = po_data[po_data["total_amount"] < APPROVAL_THRESHOLD_JPY].copy()
clusters = (
    below_threshold.groupby(["vendor_id", "vendor_name", "po_date"], as_index=False)
    .agg(
        po_count=("po_number", "count"),
        cluster_total_jpy=("total_amount", "sum"),
        average_po_amount_jpy=("total_amount", "mean"),
        material_groups=("material_group", lambda s: ", ".join(sorted(s.unique()))),
    )
)

split_clusters = clusters[(clusters["po_count"] >= MIN_CLUSTER_COUNT) & (clusters["cluster_total_jpy"] > APPROVAL_THRESHOLD_JPY)].copy()
split_clusters["cluster_id"] = [f"SPLIT-{i:04d}" for i in range(1, len(split_clusters) + 1)]
split_clusters.head(20)


## 3. Detect split patterns by vendor and date

In [ ]:
split_po_details = below_threshold.merge(
    split_clusters[["vendor_id", "po_date", "cluster_id", "po_count", "cluster_total_jpy"]],
    on=["vendor_id", "po_date"],
    how="inner",
)
split_po_details = split_po_details.sort_values(["po_date", "vendor_id", "total_amount"])
split_po_details.to_csv(OUTPUT_DIR / "split_purchase_details.csv", index=False)
split_clusters.to_csv(OUTPUT_DIR / "split_purchase_clusters.csv", index=False)
split_po_details.head(25)

## 4. Visualize split patterns over time

In [ ]:
plt.figure(figsize=(13, 6))
sns.scatterplot(
    data=split_po_details,
    x="po_date",
    y="vendor_name",
    size="total_amount",
    hue="cluster_total_jpy",
    sizes=(60, 400),
    palette="rocket_r",
)
plt.axvline(split_po_details["po_date"].min(), alpha=0)
plt.title("Timeline of Split Purchase Order Clusters")
plt.xlabel("PO date")
plt.ylabel("Vendor")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
monthly_split_trend = (
    split_po_details.assign(month=split_po_details["po_date"].dt.to_period("M").dt.to_timestamp())
    .groupby("month", as_index=False)
    .agg(split_po_count=("po_number", "count"), split_value_jpy=("total_amount", "sum"))
)

fig, ax = plt.subplots(figsize=(12, 5))
sns.barplot(data=monthly_split_trend, x="month", y="split_po_count", color="steelblue", ax=ax)
ax.set_title("Monthly Split Purchase Frequency")
ax.set_xlabel("Month")
ax.set_ylabel("Split PO count")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

## 5. Identify repeat offenders

In [ ]:
repeat_offenders = (
    split_po_details.groupby(["vendor_id", "vendor_name"], as_index=False)
    .agg(
        split_po_count=("po_number", "count"),
        split_cluster_count=("cluster_id", "nunique"),
        split_value_jpy=("total_amount", "sum"),
        avg_split_po_jpy=("total_amount", "mean"),
        first_split_date=("po_date", "min"),
        last_split_date=("po_date", "max"),
    )
    .sort_values(["split_cluster_count", "split_value_jpy"], ascending=False)
)
repeat_offenders.to_csv(OUTPUT_DIR / "split_purchase_vendor_ranking.csv", index=False)
repeat_offenders.head(15)

In [ ]:
plt.figure(figsize=(12, 6))
sns.barplot(data=repeat_offenders.head(10), y="vendor_name", x="split_cluster_count", color="darkorange")
plt.title("Vendor Ranking by Split Purchase Cluster Frequency")
plt.xlabel("Split cluster count")
plt.ylabel("Vendor")
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 5))
sns.histplot(split_po_details["total_amount"], bins=15, kde=True, color="teal")
plt.axvline(APPROVAL_THRESHOLD_JPY, color="red", linestyle="--", label="Approval threshold")
plt.title("Amount Distribution of Suspected Split POs")
plt.xlabel("PO amount (JPY)")
plt.legend()
plt.tight_layout()
plt.show()

## 6. Risk scoring by vendor

In [ ]:
repeat_offenders["frequency_score"] = repeat_offenders["split_cluster_count"].rank(pct=True) * 40
repeat_offenders["value_score"] = repeat_offenders["split_value_jpy"].rank(pct=True) * 40
repeat_offenders["proximity_score"] = (repeat_offenders["avg_split_po_jpy"] / APPROVAL_THRESHOLD_JPY).clip(upper=1) * 20
repeat_offenders["split_purchase_risk_score"] = (repeat_offenders["frequency_score"] + repeat_offenders["value_score"] + repeat_offenders["proximity_score"]).round(1)
repeat_offenders["risk_level"] = pd.cut(
    repeat_offenders["split_purchase_risk_score"],
    bins=[0, 50, 75, 100],
    labels=["Low", "Medium", "High"],
    include_lowest=True,
)
repeat_offenders.to_csv(OUTPUT_DIR / "split_purchase_risk_scorecard.csv", index=False)
repeat_offenders.head(15)